# 07 — Join dimensions and diagnose missing references

**Student exercise**

Complete the tasks in order. Each task includes an expected result, three hints, and a check. All sample data and setup code are included in this notebook.

## 1. Spark configuration

Start a fresh kernel. Use Spark 3.5.x / Scala 2.12 and Java 17. Set `STUDENT_ID` to a lowercase name containing letters, numbers or underscores. Set `SPARK_MASTER=local[2]` for a local Spark session.

In [ ]:
import os
import socket
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

master_url = os.environ.get("SPARK_MASTER", f"spark://{socket.gethostname()}:7077")
STUDENT_ID = "student01"
assert STUDENT_ID and STUDENT_ID[0].islower() and all(c in "abcdefghijklmnopqrstuvwxyz0123456789_" for c in STUDENT_ID)

spark = (
    SparkSession.builder
    .appName("Exercise-07")
    .master(master_url)
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "2")
    .config("spark.redaction.regex", "(?i)secret|password|token|access[.]?key|credential")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## 2. Sample data

Run this cell, inspect the records, and use the supplied variables in the tasks.

In [ ]:
customers = spark.createDataFrame([(1,'Asha','India'),(2,'Ben','UK'),(3,'Carla','Brazil')],
    'customer_id INT, name STRING, country STRING')
lines=spark.createDataFrame([(101,1,10,2),(102,2,20,1),(103,99,30,3),(104,3,99,1)],
 'order_id INT,customer_id INT,product_id INT,qty INT')
products=spark.createDataFrame([(10,'Mug',20.0),(20,'Bag',50.0),(30,'Notebook',10.0)],
 'product_id INT,product_name STRING,unit_price DOUBLE')

## 3. Result checks

This small function checks output values and duplicate counts. It is included here so the notebook can run on its own.

In [ ]:
def check_frame(df, columns, expected):
    assert df is not None, "Complete the task before running the check"
    assert set(columns).issubset(df.columns), "Missing output columns"
    def normalize(row):
        return tuple(round(value, 6) if isinstance(value, float) else value for value in row)
    actual = sorted(repr(normalize(row)) for row in df.select(*columns).collect())
    wanted = sorted(repr(normalize(row)) for row in expected)
    assert actual == wanted, f"Expected {wanted}, got {actual}"
    print("PASS:", columns)

## Task 1: Join known customers

Inner-join lines with customers. Return known_orders with order_id,name.

**Expected result:** Three known-customer orders.

In [ ]:
known_orders=None  # TODO: join on customer_id

**Hint 1:** An inner join removes unmatched keys, which may be unacceptable without reconciliation.

**Hint 2:** Joining by a shared string key avoids duplicate customer_id columns.

**Hint 3:** lines.join(customers,"customer_id","inner").select(...).

### Check

In [ ]:
check_frame(known_orders,['order_id','name'],[(101,'Asha'),(102,'Ben'),(104,'Carla')])

## Task 2: Find customer and product orphans

Create missing_customers and missing_products with left_anti joins.

**Expected result:** Order103 lacks customer; order104 lacks product.

In [ ]:
missing_customers=None
missing_products=None  # TODO: left_anti joins

**Hint 1:** Anti joins return left rows without a matching right key.

**Hint 2:** left_semi would return matched left rows without right-side columns.

**Hint 3:** lines.join(dimension, "key", "left_anti").

### Check

In [ ]:
check_frame(missing_customers,['order_id'],[(103,)])
check_frame(missing_products,['order_id'],[(104,)])

## Task 3: Preserve all lines with left joins

Build enriched including order_id,name,product_name,line_total; unknown fields remain null.

**Expected result:** All four original lines retained.

In [ ]:
enriched=None  # TODO: two left joins and qty*unit_price

**Hint 1:** Left joins preserve facts only if the dimensions have unique join keys.

**Hint 2:** Unknown price should not silently become zero revenue.

**Hint 3:** Chain left joins, derive line_total, and explicitly select output columns.

### Check

In [ ]:
check_frame(enriched,['order_id','name','product_name','line_total'],[(101,'Asha','Mug',40.0),(102,'Ben','Bag',50.0),(103,None,'Notebook',30.0),(104,'Carla',None,None)])

## More practice

Reproduce the enriched result in SQL with aliases; duplicate a dimension key and measure join fan-out.

## Common mistakes

Joining nonunique dimensions multiplies facts. A filter on a right column after a left join can remove unmatched rows.

## Finish

Stop Spark before starting another notebook. For write exercises, run setup again before repeating append tasks. S3 data remains available after stopping Spark.

In [ ]:
spark.stop()